# Chapter 9. AHP, ANP, Fuzzy AHP, and IPA

*Starting With What You Have: A Quantitative Field Guide for Urban Research in Data-Scarce Settings*

Runs in a browser with no installation. Open in Google Colab and choose Runtime, then Run all.


## Step 0. Installation

For fieldwork the spreadsheet workbook is the better route, since respondents see their own consistency ratio immediately. Use this notebook for aggregation, screening, and sensitivity analysis.

In [ ]:
!pip install -q pandas numpy matplotlib

## Step 1. Load the judgements

25 respondents comparing five criteria. Only the upper triangle is asked; the lower triangle follows from the reciprocal property.

In [ ]:
import numpy as np
import pandas as pd
import mcdm_toolkit as m

LABELS = ["water", "sanitation", "roads", "electricity", "tenure"]
mats = np.load("data/matrices.npy")
n = len(LABELS)
print("respondents:", mats.shape[0], "| criteria:", n)
print("comparisons per respondent:", n * (n - 1) // 2)

## Step 2. Consistency screening, before aggregation

A ratio above 0.10 indicates contradiction at more than ten per cent of the level a random response would show. Above 1.0 the response is more contradictory than answering at random.

In [ ]:
scr = m.screen_respondents(mats, cr_max=0.10)
print(f"total {len(scr)} | pass {int(scr.keep.sum())} | fail {int((~scr.keep).sum())}")
print("failing CRs:", [round(x, 3) for x in sorted(scr.CR[~scr.keep])])

## Step 3. Group aggregation by geometric mean

**Required, not preferred.** Only the geometric mean preserves reciprocity: combining 3 and 1/3 arithmetically gives 1.667 in both directions, which violates AHP's own premise.

In [ ]:
G = m.aggregate_judgments(mats[scr.keep.values])
res = m.ahp(G, LABELS)
for k, v in sorted(res["weights"].items(), key=lambda x: -x[1]):
    print(f"  {k:12s} {v:.4f}")
print("group CR =", round(res["CR"], 4))

## Step 4. What happens if you skip the screening

Leaving inconsistent responses in flattens the distribution, because contradictory judgements act as noise pulling every item towards equality.

In [ ]:
w_keep = m.ahp(G, LABELS)["weights"]
w_all = m.ahp(m.aggregate_judgments(mats), LABELS)["weights"]
cmp = pd.DataFrame({"screened": w_keep, "unscreened": w_all})
cmp["difference"] = (cmp.screened - cmp.unscreened).round(4)
print(cmp.round(4).to_string())

## Step 5. Eigenvector against geometric mean

The two agree to the fourth decimal place. That agreement is what makes the spreadsheet workbook possible, since the geometric mean can be written as a formula and the eigenvector cannot.

In [ ]:
we = m.priority_vector(G, "eigen")
wg = m.priority_vector(G, "gmean")
print(pd.DataFrame({"eigenvector": we, "geometric mean": wg,
                    "difference": np.abs(we - wg)}, index=LABELS).round(5).to_string())

## Step 6. Fuzzy AHP, retaining the uncertainty

If the intervals of the first and second ranked criteria **overlap**, differentiating budget allocation on that ranking is not defensible.

In [ ]:
fz = m.fuzzy_ahp(G, LABELS)
print("defuzzified weights and fuzzy intervals:")
for k, v in sorted(fz["weights"].items(), key=lambda x: -x[1]):
    lo, mid, hi = fz["fuzzy_weights"][k]
    print(f"  {k:12s} {v:.4f}   interval ({lo:.3f}, {mid:.3f}, {hi:.3f})")

## Step 7. IPA, crossing importance with performance

The 'concentrate here' quadrant, high importance and low performance, is the policy conclusion.

In [ ]:
perf = pd.read_csv("data/performance_survey.csv")[LABELS].mean().values
imp = np.array([w_keep[k] for k in LABELS])
ipa, (imp_mean, perf_mean) = m.ipa(imp, perf, LABELS)
print(ipa.to_string(index=False))
print()
print(f"quadrant boundaries: importance {imp_mean:.3f}, performance {perf_mean:.3f}")

## Step 8. ANP, when criteria are entangled

Before using it, ask whether you can **justify** the influence values. They are themselves judgements, and filling them without a basis adds complexity while weakening the result.

In [ ]:
supermatrix = np.array([
    [0.00, 0.30, 0.20, 0.10, 0.15],
    [0.25, 0.00, 0.25, 0.10, 0.15],
    [0.20, 0.20, 0.00, 0.20, 0.20],
    [0.15, 0.15, 0.25, 0.00, 0.20],
    [0.40, 0.35, 0.30, 0.60, 0.30],
])
anp = m.anp(supermatrix, LABELS)
for k, v in sorted(anp["weights"].items(), key=lambda x: -x[1]):
    print(f"  {k:12s} {v:.4f}")
print("iterations to convergence:", anp["iterations"])

---

**What to do next.** Compare against Section 9.4. Report the composition of your respondent group, and check whether weights differ by group: that difference is frequently a finding in itself.